# Week 7 — Block 2: Guided Demo (Text Data)

**DATS 6401 · Visualization of Complex Data**

~30 min:

1. Counts: sorted bars vs. the word cloud, side by side (~8 min)
2. TF–IDF flips frequent → distinctive (~7 min)
3. **BERTopic** on a real corpus (~10 min — requires install/download, run live in class)
4. Embedding projection (~5 min)

Parts 1, 2, 4 run fully offline on the inline corpus below; Part 3's cells are marked.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

docs = [
    "the model trains on data and the data shapes the model",
    "visualization turns data into pictures people can read",
    "a topic model finds themes hiding in a corpus of text",
    "embeddings place similar text near similar text in space",
    "people read pictures faster than they read tables of data",
    "text needs a transform before any visualization can begin",
    "a corpus of documents becomes a matrix of token counts",
    "the matrix view of text loses word order but gains structure",
]
print(len(docs), 'documents')

## Part 1 — Counts: the honest view vs. the poster

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(stop_words="english")
counts = cv.fit_transform(docs)
freq = pd.Series(counts.sum(axis=0).A1, index=cv.get_feature_names_out())

from wordcloud import WordCloud
wc = WordCloud(width=420, height=280, background_color="white",
               colormap="GnBu").generate_from_frequencies(freq.to_dict())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
freq.sort_values().tail(10).plot.barh(ax=axes[0], color="#2E6E8E")
axes[0].set_title("Sorted bars: is #3 bigger than #5? Instant.")
axes[1].imshow(wc); axes[1].axis("off")
axes[1].set_title("Word cloud: same data — now rank #3 vs #5")
plt.tight_layout(); plt.show()

**Run the Week 2 test live:** ask the room to rank terms 3–5 from the cloud, then reveal the bars. The gap *is* the area-vs-position ranking.

## Part 2 — TF–IDF: frequent → distinctive

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfv = TfidfVectorizer(stop_words="english")
T = tfv.fit_transform(docs)

d = 3   # the embeddings doc
raw = pd.Series(counts[d].toarray()[0], index=cv.get_feature_names_out())
tfd = pd.Series(T[d].toarray()[0], index=tfv.get_feature_names_out())

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4))
raw[raw > 0].sort_values().plot.barh(ax=axes[0], color="#5a6672")
axes[0].set_title("Doc 4, RAW counts")
tfd[tfd > 0].sort_values().plot.barh(ax=axes[1], color="#2E6E8E")
axes[1].set_title("Same doc, TF–IDF: corpus-common words sink")
plt.tight_layout(); plt.show()

## Part 3 — BERTopic (⚠️ live-in-class cells)

The cells below need `pip install bertopic` and download a sentence-transformer on first run (~100 MB) — **internet required**. On a real corpus (e.g., a few hundred news headlines or course feedback), run:

```python
from bertopic import BERTopic

topic_model = BERTopic(min_topic_size=5)
topics, probs = topic_model.fit_transform(real_docs)

topic_model.get_topic_info()          # the table: note topic -1, the OUTLIER bin
topic_model.visualize_topics()        # the topic map
topic_model.visualize_barchart()      # c-TF-IDF terms per topic
```

**Narrate while it fits:** embed → UMAP → HDBSCAN → c-TF-IDF (the Block 1 pipeline). Then interrogate the outlier bin's size *first* — it's the honesty number.

## Part 4 — Embeddings as geometry (offline stand-in)

In [ ]:
from sklearn.decomposition import PCA

Xt = T.toarray()
pcs = PCA(n_components=2).fit_transform(Xt)
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(pcs[:, 0], pcs[:, 1], s=130, color="#2E6E8E")
for i, (x_, y_) in enumerate(pcs):
    ax.annotate(f"doc {i+1}", (x_, y_), textcoords="offset points", xytext=(8, 5), fontsize=9)
ax.set_title("TF–IDF vectors → PCA: neighbors share vocabulary")
plt.show()

**Caption discipline reminder:** with real embeddings you'd swap PCA → UMAP and the caption must name embedding, projection, parameters.

## Wrap-up → Block 3

Pipeline: **counts → distinctive → topics → geometry**, honesty at each hop. Your corpus next.